In [1]:
import sys
sys.path.insert(0, '../src')
from models import load_model
from utils import *
from tversky_utils import *

config = parse_config('../configs/tversky_proj.yaml')
# latent dim 6, fbank size 128

model = load_model(config, "../results/tversky_proj_gridrobot/tversky_proj_1784130650.pth")
data = load_data(config)
all_trajs = data["trajs"]
all_feats = data["features"]

loading data: gridrobot_1960


# prediction
* sample high feature value trajs and label them "hi", low feature value trajs and label them "lo"
* find one (hi,lo) pair and do maxmin, minmax queries
* take another (hi, lo pair) and for each traj: compute similarity to hi features, similarity to lo features (or salience?)
    * print results

In [2]:
import numpy as np
import itertools
import torch

In [3]:
all_feats.shape # laptop, table

(1960, 2)

In [ ]:
# using laptop (feature idx 0) because it is more query-able than upright, 
# see experiments/002-tversky-query-eval/figs/tversky_proj_fbank_size.png
hi_indices = np.where(all_feats[:,0] == np.max(all_feats[:,0]))[0]
lo_indices = np.where(all_feats[:,0] == np.min(all_feats[:,0]))[0]
print(f"{len(hi_indices)} hi indices, {len(lo_indices)} lo indices")
hi_trajs = all_trajs[hi_indices]
lo_trajs = all_trajs[lo_indices]

56 hi indices, 448 lo indices


In [5]:
pairs = np.array(list(itertools.product(hi_indices, lo_indices)))    # all (max_traj, min_traj) pairs)
pairs.shape

(25088, 2)

In [39]:
TOP_FEATURE_COUNT = 128 # this is the feature bank size, so query will retrieve all salient features
TOP_RESULT_COUNT = 5
feature_bank = model.encoder[0].feature_bank.weight.detach()  # (F, D)
trajs_t = torch.as_tensor(all_trajs, dtype=torch.float32)
centered_trajs = (trajs_t - trajs_t.mean(0)).detach()       # (N, D)
def run_query(a_idx, b_idx, top_feature_count=TOP_FEATURE_COUNT, top_result_count=TOP_RESULT_COUNT):
    """s(a) - s(b): retrieve instances salient for a's features but not b's."""
    return retrieve_semantic_expression(
        instance_vectors=centered_trajs,
        feature_bank=feature_bank,
        expression=f"s({a_idx})-s({b_idx})",
        top_feature_count=top_feature_count,
        top_result_count=top_result_count,
    )



In [7]:
pair = pairs[0]
m,n = pair
res_maxmin = run_query(m, n)   # s(max) - s(min)
res_minmax = run_query(n, m)   # s(min) - s(max)

In [8]:
res_maxmin

{'expression': 's(29)-s(1)',
 'query_item_ixes': [29, 1],
 'feature_count': 51,
 'top_instances': [{'item_ix': 128,
   'salience': 86.81338500976562,
   'measure': 86.81338500976562},
  {'item_ix': 29, 'salience': 83.70231628417969, 'measure': 83.70231628417969},
  {'item_ix': 878,
   'salience': 80.59123992919922,
   'measure': 80.59123992919922},
  {'item_ix': 293, 'salience': 80.199951171875, 'measure': 79.64033508300781},
  {'item_ix': 885,
   'salience': 78.05712127685547,
   'measure': 77.90460205078125}],
 'semantic_features': {0,
  1,
  4,
  10,
  11,
  15,
  17,
  19,
  20,
  22,
  23,
  25,
  26,
  27,
  28,
  32,
  41,
  44,
  45,
  46,
  50,
  51,
  58,
  61,
  62,
  64,
  67,
  68,
  69,
  70,
  73,
  75,
  76,
  79,
  81,
  86,
  87,
  88,
  89,
  93,
  98,
  99,
  101,
  107,
  108,
  110,
  115,
  118,
  121,
  125,
  126}}

In [9]:
res_minmax

{'expression': 's(1)-s(29)',
 'query_item_ixes': [1, 29],
 'feature_count': 19,
 'top_instances': [{'item_ix': 1408,
   'salience': 25.760372161865234,
   'measure': 25.760372161865234},
  {'item_ix': 822,
   'salience': 26.527915954589844,
   'measure': 25.4534854888916},
  {'item_ix': 1545,
   'salience': 25.22469711303711,
   'measure': 25.22469711303711},
  {'item_ix': 1597,
   'salience': 26.063980102539062,
   'measure': 24.917810440063477},
  {'item_ix': 1300,
   'salience': 24.689023971557617,
   'measure': 24.689023971557617}],
 'semantic_features': {2,
  9,
  13,
  16,
  30,
  53,
  54,
  59,
  71,
  72,
  78,
  82,
  85,
  91,
  97,
  103,
  104,
  114,
  117}}

In [10]:
res_minmax['semantic_features']

{2, 9, 13, 16, 30, 53, 54, 59, 71, 72, 78, 82, 85, 91, 97, 103, 104, 114, 117}

In [11]:
test_pair = pairs[-1]
test_hi, test_lo = test_pair

In [12]:
import torch.nn.functional as F

def selective_salience(test_traj_idx, query_res):
    # get resulting features of this query
    semantic_features = query_res['semantic_features']
    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    centered_test_traj = centered_trajs[test_traj_idx]
    dot = centered_test_traj @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    # p_measures  = dot.sum(dim=1)                        # (N,)
    return p_saliences

In [13]:
maxmin_salience = selective_salience(torch.tensor(test_pair), res_maxmin)
minmax_salience = selective_salience(torch.tensor(test_pair), res_minmax)

print("                | hi traj  |  lo traj")
print(f"maxmin salience | {maxmin_salience[0]} | {maxmin_salience[1]}")
print(f"minmax salience | {minmax_salience[0]} | {minmax_salience[1]}")

                | hi traj  |  lo traj
maxmin salience | 14.339900970458984 | 18.17135238647461
minmax salience | 4.678924560546875 | 21.282440185546875


^ this is promising - high trajectory has greater salience with `res_maxmin` (which was hi - lo) and low trajectory has greater salience with `res_minmax` (which was lo - hi)!!

prediction:
take selective salience with res_maxmin

In [14]:
def tversky_predict(test_traj, hi_res, lo_res):
    hi_salience = selective_salience(torch.tensor([test_traj]), hi_res)
    lo_salience = selective_salience(torch.tensor([test_traj]), lo_res)

    # TODO uncertainty measurement? softmax or sigmoid or something else...

    if hi_salience > lo_salience:
        return "hi"
    return "lo"


In [15]:
tversky_predict(test_lo, res_maxmin, res_minmax)

'lo'

In [16]:

tversky_predict(test_hi, res_maxmin, res_minmax)

'hi'

# query narrowing

In [17]:
pairs.shape

(25088, 2)

In [18]:
import torch.nn.functional as F

def selective_salience(test_traj_idx, semantic_features):
    if len(semantic_features) == 0:
        # no features survived narrowing -> no evidence either way
        return torch.zeros(len(test_traj_idx))
    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features), dtype=torch.long)
    )
    centered_test_traj = centered_trajs[test_traj_idx]
    dot = centered_test_traj @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=-1) / len(semantic_features)   # (N,)
    return p_saliences


def tversky_predict(test_traj, hi_feats, lo_feats):
    hi_salience = selective_salience(torch.tensor([test_traj]), hi_feats)
    lo_salience = selective_salience(torch.tensor([test_traj]), lo_feats)

    # TODO uncertainty measurement? softmax or sigmoid or something else...

    if hi_salience > lo_salience:
        return "hi"
    return "lo"


In [19]:
TRAIN_TEST_SPLIT = 0.8
n = len(pairs)
rng = np.random.default_rng(seed=0)
perm = rng.permutation(n)
n_train = int(n * TRAIN_TEST_SPLIT)
train_idx, test_idx = perm[:n_train], perm[n_train:]
train_pairs = pairs[train_idx]
test_pairs = pairs[test_idx]

In [20]:
from collections import Counter

N_QUERIES = n_train
VOTE_FRAC = 0.5   # keep features seen in >= this fraction of queries so far
                  # 1.0 == intersection, ~0 == union

hi_votes, lo_votes = Counter(), Counter()
accs = []
for i in range(N_QUERIES):
    hi, lo = train_pairs[i]
    hi_votes.update(run_query(hi, lo)["semantic_features"])
    lo_votes.update(run_query(lo, hi)["semantic_features"])

    thresh = VOTE_FRAC * (i + 1)
    hi_feats = {f for f, c in hi_votes.items() if c >= thresh}
    lo_feats = {f for f, c in lo_votes.items() if c >= thresh}

    # test (counters reset each round -> accuracy of *this* feature set)
    total_test = 0
    test_correct = 0
    for test_pair in test_pairs:
        # kinda weird but since eval is per traj, we have two predictions per pair (one per traj)
        hi_test, lo_test = test_pair
        hi_predict = tversky_predict(hi_test, hi_feats, lo_feats)
        if hi_predict == "hi":
            test_correct += 1
        lo_predict= tversky_predict(lo_test, hi_feats, lo_feats)
        if lo_predict == "lo":
            test_correct += 1
        total_test += 2
    acc = test_correct / total_test
    accs.append(acc)
    print(f"test accuracy after {i + 1} queries: {acc:.4f}  "
          f"(|hi_feats|={len(hi_feats)}, |lo_feats|={len(lo_feats)})")

test accuracy after 1 queries: 0.4995  (|hi_feats|=39, |lo_feats|=29)
test accuracy after 2 queries: 0.5196  (|hi_feats|=63, |lo_feats|=51)
test accuracy after 3 queries: 0.3636  (|hi_feats|=16, |lo_feats|=10)
test accuracy after 4 queries: 0.4739  (|hi_feats|=25, |lo_feats|=26)
test accuracy after 5 queries: 0.4776  (|hi_feats|=6, |lo_feats|=11)
test accuracy after 6 queries: 0.4862  (|hi_feats|=9, |lo_feats|=11)
test accuracy after 7 queries: 0.4339  (|hi_feats|=3, |lo_feats|=2)
test accuracy after 8 queries: 0.5322  (|hi_feats|=9, |lo_feats|=5)
test accuracy after 9 queries: 0.5040  (|hi_feats|=1, |lo_feats|=0)
test accuracy after 10 queries: 0.4969  (|hi_feats|=4, |lo_feats|=1)
test accuracy after 11 queries: 0.5000  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 12 queries: 0.5000  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 13 queries: 0.5000  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 14 queries: 0.5000  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 15 queries: 0

KeyboardInterrupt: 

:^(

# trying again but train-test splitting on trajs, not pairs

In [26]:

TRAIN_TEST_SPLIT = 0.8
n = len(all_trajs)
rng = np.random.default_rng(seed=0)
perm = rng.permutation(n)
n_train = int(n * TRAIN_TEST_SPLIT)
train_idx, test_idx = perm[:n_train], perm[n_train:]
train_trajs = all_trajs[train_idx]
train_feats = all_feats[train_idx]
test_trajs = all_trajs[test_idx]
test_feats = all_feats[test_idx]


hi_indices = np.where(train_feats[:,0] == np.max(train_feats[:,0]))[0]
lo_indices = np.where(train_feats[:,0] == np.min(train_feats[:,0]))[0]
hi_trajs = train_trajs[hi_indices]
lo_trajs = train_trajs[lo_indices]
train_pairs = np.array(list(itertools.product(hi_indices, lo_indices)))    # all (max_traj, min_traj) pairs)
train_pairs.shape

(14637, 2)

In [27]:
test_feats

array([[0.59050898, 0.33333333],
       [0.21850801, 1.        ],
       [0.09050898, 1.        ],
       [0.5       , 0.66666667],
       [0.30901699, 0.66666667],
       [0.39952598, 0.66666667],
       [0.        , 0.33333333],
       [0.09050898, 1.        ],
       [0.        , 1.        ],
       [0.        , 0.        ],
       [0.09050898, 0.33333333],
       [0.5       , 0.66666667],
       [0.78149199, 0.33333333],
       [0.09050898, 0.66666667],
       [0.5       , 0.33333333],
       [0.30901699, 0.66666667],
       [0.78149199, 1.        ],
       [0.18101796, 0.33333333],
       [0.        , 1.        ],
       [0.09050898, 1.        ],
       [0.09050898, 1.        ],
       [0.        , 0.66666667],
       [0.09050898, 0.33333333],
       [0.30901699, 1.        ],
       [0.59050898, 0.66666667],
       [0.30901699, 0.66666667],
       [0.21850801, 0.33333333],
       [1.        , 0.33333333],
       [0.        , 1.        ],
       [0.30901699, 0.66666667],
       [0.

In [23]:
n_train = train_pairs.shape[0]

In [ ]:
from collections import Counter

N_QUERIES = n_train
VOTE_FRAC = 0.5   # keep features seen in >= this fraction of queries so far
                  # 1.0 == intersection, ~0 == union

hi_votes, lo_votes = Counter(), Counter()
accs = []
for i in range(N_QUERIES):
    hi, lo = train_pairs[i]
    hi_votes.update(run_query(hi, lo, top_feature_count=128)["semantic_features"])
    lo_votes.update(run_query(lo, hi, top_feature_count=128)["semantic_features"])

    thresh = VOTE_FRAC * (i + 1)
    hi_feats = {f for f, c in hi_votes.items() if c >= thresh}
    lo_feats = {f for f, c in lo_votes.items() if c >= thresh}

    # test (counters reset each round -> accuracy of *this* feature set)
    total_test = 0
    test_correct = 0
    for test_i, test_feat in zip(test_idx, test_feats):
        prediction = tversky_predict(int(test_i), hi_feats, lo_feats)
        if prediction == "hi" and test_feat[0] > 0.5:
            test_correct += 1
        if prediction == "lo" and test_feat[0] <= 0.5:
            test_correct += 1
        total_test += 1
    acc = test_correct / total_test
    accs.append(acc)
    print(f"test accuracy after {i + 1} queries: {acc:.4f}  "
          f"(|hi_feats|={len(hi_feats)}, |lo_feats|={len(lo_feats)})")

test accuracy after 1 queries: 0.6842  (|hi_feats|=23, |lo_feats|=22)
test accuracy after 2 queries: 0.5789  (|hi_feats|=9, |lo_feats|=14)
test accuracy after 3 queries: 0.5263  (|hi_feats|=7, |lo_feats|=6)
test accuracy after 4 queries: 0.5263  (|hi_feats|=4, |lo_feats|=1)
test accuracy after 5 queries: 0.6842  (|hi_feats|=1, |lo_feats|=1)
test accuracy after 6 queries: 0.5789  (|hi_feats|=1, |lo_feats|=0)
test accuracy after 7 queries: 0.5789  (|hi_feats|=1, |lo_feats|=0)
test accuracy after 8 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 9 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 10 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 11 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 12 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 13 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 14 queries: 0.6842  (|hi_feats|=0, |lo_feats|=0)
test accuracy after 15 queries: 0.6842  

KeyError: 'semantic_features'